# iRacing season results analysis

Reads every exported `eventresult-*.json` file in the `data` folder. Each subfolder is a season, for example `data/Europe/`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib inline

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 50)

 |->
 Input cell:
 - `DATA_DIR`: the folder holding the exports. Leave it as `None` to use the `data` folder next to this notebook.
 - `SEASON`: a subfolder name (for example `'Europe'`), or `None` for all seasons.
 - `INCLUDE_AI`: whether AI drivers count in the standings.

In [ ]:
DATA_DIR = None   # e.g. r'C:\Users\andre\OneDrive - Pavtech Solutions Pty Ltd\Documents\Iracing-Seasons\data'
SEASON = 'Europe'
INCLUDE_AI = True

Load every result file:

In [ ]:
data_dir = Path(DATA_DIR) if DATA_DIR else Path.cwd() / 'data'
search_dir = data_dir / SEASON if SEASON else data_dir

def to_seconds(t):
    """iRacing times are in 1/10000 s; -1 means no time."""
    return t / 10000.0 if t is not None and t > 0 else np.nan

def lap_str(t):
    if pd.isna(t):
        return ''
    minutes = int(t // 60)
    return f"{minutes}:{t - minutes * 60:06.3f}"

events = []
for path in sorted(search_dir.rglob('eventresult-*.json')):
    with open(path, encoding='utf-8') as f:
        raw = json.load(f)
    event = raw.get('data', raw)   # exports are wrapped as {"type": "event_result", "data": {...}}
    event['_season'] = path.parent.name
    event['_file'] = path.name
    events.append(event)

events.sort(key=lambda e: e['start_time'])
print(f"Loaded {len(events)} events from {search_dir}")
for i, e in enumerate(events, 1):
    print(f"  Round {i}: {e['start_time'][:10]}  {e['track']['track_name']} ({e['track'].get('config_name') or ''})  subsession {e['subsession_id']}")

Build one row per driver per race, with their qualifying lap added:

In [ ]:
rows = []
for rnd, e in enumerate(events, 1):
    sessions = {s['simsession_number']: s for s in e['session_results']}
    race = sessions.get(0)
    if race is None:
        continue
    qual_best = {}
    for s in e['session_results']:
        if s['simsession_type_name'].endswith('Qualifying') or 'Qual' in s['simsession_type_name']:
            for r in s['results']:
                qual_best[r['cust_id']] = to_seconds(r['best_lap_time'])

    track = e['track']['track_name']
    for r in race['results']:
        rows.append({
            'Season': e['_season'],
            'Round': rnd,
            'Date': e['start_time'][:10],
            'Track': track,
            'Subsession': e['subsession_id'],
            'Driver': r['display_name'],
            'Cust ID': r['cust_id'],
            'AI': r.get('ai', False),
            'Car': r['car_name'],
            'Class': r.get('car_class_short_name'),
            'Start': r['starting_position'] + 1,
            'Finish': r['finish_position'] + 1,
            'Class finish': r['finish_position_in_class'] + 1,
            'Positions gained': r['starting_position'] - r['finish_position'],
            'Laps': r['laps_complete'],
            'Laps led': r['laps_lead'],
            'Best lap': to_seconds(r['best_lap_time']),
            'Best lap num': r['best_lap_num'],
            'Average lap': to_seconds(r['average_lap']),
            'Qual lap': qual_best.get(r['cust_id'], np.nan),
            'Incidents': r['incidents'],
            'Points': r.get('league_points', r.get('champ_points', 0)),
            'Status': r['reason_out'],
            'iRating change': (r['newi_rating'] - r['oldi_rating']) if r['oldi_rating'] > 0 else np.nan,
        })

results = pd.DataFrame(rows)
if not INCLUDE_AI:
    results = results[~results['AI']]

# Gap of each driver's best lap to the fastest lap of that race
results['Best lap gap %'] = (results['Best lap'] / results.groupby('Round')['Best lap'].transform('min') - 1) * 100
results.head(30)

Race summaries:

In [ ]:
def summary(group):
    winner = group.loc[group['Finish'].idxmin()]
    fastest = group.loc[group['Best lap'].idxmin()]
    pole = group.loc[group['Start'].idxmin()]
    return pd.Series({
        'Date': winner['Date'],
        'Track': winner['Track'],
        'Starters': len(group),
        'Winner': winner['Driver'],
        'Pole': pole['Driver'],
        'Fastest lap': f"{fastest['Driver']} ({lap_str(fastest['Best lap'])})",
        'Total incidents': group['Incidents'].sum(),
    })

race_summary = results.groupby('Round').apply(summary, include_groups=False)
race_summary

Championship standings:

In [ ]:
standings = results.groupby('Driver').agg(
    Points=('Points', 'sum'),
    Races=('Round', 'nunique'),
    Wins=('Finish', lambda s: (s == 1).sum()),
    Podiums=('Finish', lambda s: (s <= 3).sum()),
    Best=('Finish', 'min'),
    Avg_finish=('Finish', 'mean'),
    Avg_gained=('Positions gained', 'mean'),
    Laps_led=('Laps led', 'sum'),
    Incidents=('Incidents', 'sum'),
).sort_values(['Points', 'Wins'], ascending=False)

standings['Inc per race'] = standings['Incidents'] / standings['Races']
standings.insert(0, 'Pos', range(1, len(standings) + 1))
standings.round(2)

 |->
 Input cell:
 - Number of drivers (top of the standings) to show in the charts

In [ ]:
TOP_N = 10

Points progression:

In [ ]:
top = standings.head(TOP_N).index
points = results[results['Driver'].isin(top)].pivot_table(index='Round', columns='Driver', values='Points', aggfunc='sum')
cumulative = points.reindex(range(1, results['Round'].max() + 1)).fillna(0).cumsum()[top]

track_labels = [f"R{r}\n{t}" for r, t in results.groupby('Round')['Track'].first().items()]

plt.rcParams['figure.figsize'] = [14, 7]
for driver in top:
    plt.plot(cumulative.index, cumulative[driver], marker='o', label=driver)
plt.xticks(cumulative.index, track_labels)
plt.ylabel('Points')
plt.title('Championship points')
plt.legend(ncol=2)
plt.grid(True, alpha=0.3)
plt.show()

Finishing position by round:

In [ ]:
finish = results[results['Driver'].isin(top)].pivot_table(index='Round', columns='Driver', values='Finish')[top]

for driver in top:
    plt.plot(finish.index, finish[driver], marker='o', label=driver)
plt.xticks(finish.index, track_labels)
plt.gca().invert_yaxis()
plt.ylabel('Finish position')
plt.title('Finishing position')
plt.legend(ncol=2)
plt.grid(True, alpha=0.3)
plt.show()

Pace: best lap gap to the fastest lap of each race (%):

In [ ]:
pace = results.pivot_table(index='Driver', columns='Round', values='Best lap gap %').loc[standings.index]
pace.columns = [f"R{c} {t}" for c, t in results.groupby('Round')['Track'].first().items()]
pace['Season avg'] = pace.mean(axis=1)
pace.round(2).style.background_gradient(cmap='RdYlGn_r', axis=None).format('{:.2f}', na_rep='')

Best lap times per race:

In [ ]:
best_laps = results.pivot_table(index='Driver', columns='Round', values='Best lap').loc[standings.index]
best_laps.columns = [f"R{c} {t}" for c, t in results.groupby('Round')['Track'].first().items()]
best_laps.map(lap_str)

 |->
 Input cell:
 - Driver to look at in detail

In [ ]:
DRIVER = 'Tony Pajvancic'

In [ ]:
d = results[results['Driver'] == DRIVER].copy()
for col in ['Best lap', 'Average lap', 'Qual lap']:
    d[col] = d[col].map(lap_str)
d[['Round', 'Date', 'Track', 'Car', 'Start', 'Finish', 'Positions gained', 'Laps led',
   'Qual lap', 'Best lap', 'Average lap', 'Best lap gap %', 'Incidents', 'Points', 'iRating change', 'Status']].round(2)

### Note: lap-by-lap analysis

The event result exports only include each driver's best and average lap, not individual laps, so the gap graph, lap-time histograms and pit-stop analysis from the original notebook can't be built from them. They need per-car lap data, which is a separate request in iRacing.